In [ ]:
import os, sys
import pandas as pd
from tqdm.auto import tqdm
import gc
import random
import itertools

sys.path.append(os.path.join(os.getcwd(), '../src'))
from utils import extract_column_names_from_ctl_file


## Defining Paths and Limits

Configure input paths and set maximum limits for parent and grandparent concepts.


In [ ]:
UMLS_VERSION = "2023AA"
UMLS_PATH = ""
MAX_NOPARENTS = 30
MAX_PARENTS = 30
MAX_GRAND_PARENTS = 30

## Loading and Filtering Concepts

Load UMLS concepts and filter to keep only English and Spanish terms.

In [ ]:
with open(os.path.join(UMLS_PATH, "MRCONSO.RRF"), "r") as f:
    lines = f.readlines()
print (len(lines))

cleaned = []
for l in tqdm(lines):
    lst = l.rstrip("\n").split("|")
    cui, lang, synonym = lst[0], lst[1], lst[14]
    if lang not in ["ENG", "SPA"]: continue 
    cleaned.append(lst[0:-1])
print (len(cleaned))

## Structuring Concepts

Organize the information into a DataFrame, removing duplicates and keeping key columns for hierarchical relations.


In [ ]:
colnames = extract_column_names_from_ctl_file(os.path.join(UMLS_PATH, "MRCONSO.ctl"))
conso_es_en_df = pd.DataFrame(cleaned, columns=colnames)[["CUI","STR","LAT","TS","STT","ISPREF","AUI"]]
conso_es_en_df.drop_duplicates(inplace=True)
conso_es_en_df.head()

In [ ]:
del cleaned, lines
gc.collect()

In [ ]:
conso_en_df = conso_es_en_df[conso_es_en_df.LAT=="ENG"].reset_index(drop=True)
conso_es_df = conso_es_en_df[conso_es_en_df.LAT=="SPA"].reset_index(drop=True)

In [ ]:
conso_en_df.shape, conso_es_df.shape

In [ ]:
dict_AUI_CUI = dict(zip(conso_es_en_df.AUI.to_list(), conso_es_en_df.CUI.to_list()))

In [ ]:
del conso_es_en_df
gc.collect()

## Including TradeNames

This section adds **TradeNames** for each concept, providing additional term variants that may be relevant in clinical contexts.

It is important to note that **TradeNames are only available in the English version of UMLS**, so this step is performed exclusively on English concepts. This enrichment allows models to capture commercial names alongside standardized terminologies, enhancing their semantic representation.


In [ ]:
with open(os.path.join(UMLS_PATH, "MRREL.RRF"), "r") as f:
    lines = f.readlines()
print (len(lines))

cleaned = []
for l in tqdm(lines):
    lst = l.rstrip("\n").split("|")
    rel_atribute = lst[7]
    if rel_atribute != "has_tradename": continue 
    cleaned.append(lst[0:-1])
print (len(cleaned))

In [ ]:
colnames = extract_column_names_from_ctl_file(os.path.join(UMLS_PATH, "MRREL.ctl"))
rel_ht_df = pd.DataFrame(cleaned, columns=colnames)
rel_ht_df.drop_duplicates(inplace=True)
rel_ht_df.head()

In [ ]:
rel_ht_df.shape

In [ ]:
del cleaned, lines
gc.collect()

In [ ]:
tradenames_CUIs = rel_ht_df.AUI1.to_list()
conso_tradenames = conso_en_df[conso_en_df.AUI.isin(tradenames_CUIs)]



tradenames_es = pd.DataFrame()
tradenames_es['CUI'] = rel_ht_df['CUI2']
tradenames_es['STR'] = ''
tradenames_es['LAT'] = 'ENG'
tradenames_es['TS'] = 'S'
tradenames_es['STT'] = ''
tradenames_es['ISPREF'] = ''
tradenames_es['AUI'] = rel_ht_df['AUI1']

unique_tradenames_rows = conso_tradenames.groupby(['AUI', 'STR']).filter(lambda x: len(x) == 1)
trade_aui_str = unique_tradenames_rows.groupby('AUI')['STR'].apply(list).to_dict()
trade_aui_stt = unique_tradenames_rows.groupby('AUI')['STT'].apply(list).to_dict()
trade_aui_ispref = unique_tradenames_rows.groupby('AUI')['ISPREF'].apply(list).to_dict()
tradenames_es["STR"] = tradenames_es.AUI.map(trade_aui_str)
tradenames_es["STT"] = tradenames_es.AUI.map(trade_aui_stt)
tradenames_es["ISPREF"] = tradenames_es.AUI.map(trade_aui_ispref)
tradenames_es["STR"] = tradenames_es.STR.explode()
tradenames_es["STT"] = tradenames_es.STT.explode()
tradenames_es["ISPREF"] = tradenames_es.ISPREF.explode()
conso_es_td_df = pd.concat([conso_es_df,tradenames_es]).reset_index(drop=True).copy()


In [ ]:
conso_es_td_df.shape 

In [ ]:
del rel_ht_df, conso_en_df, conso_tradenames, tradenames_es
gc.collect()

## Loading the MRHIER File to Extract Hierarchical Relations

In this section, the **MRHIER.RRF** file is loaded, which contains the **hierarchical relations** between UMLS concepts. This file will be used to identify the **parents** and **grandparents** of each concept.

It is important to note that **MRHIER does not directly provide the parent or grandparent CUI (Concept Unique Identifier)**. Instead, it provides the **AUI (Atom Unique Identifier)**, which refers to specific terms or synonyms. Therefore, a **manual mapping from AUI to CUI** is required, using the previously loaded data from the **MRCONSO.RRF** file.

This step is crucial to building the correct hierarchical relations that will enrich the knowledge of the Representation LLMs.


In [ ]:
with open(os.path.join(UMLS_PATH, "MRHIER.RRF"), "r") as f:
    lines = f.readlines()
print (len(lines))


In [ ]:
cleaned = []
for l in tqdm(lines):
    lst = l.rstrip("\n").split("|")
    cleaned.append(lst[0:-1])
print (len(cleaned))

In [ ]:
colnames = extract_column_names_from_ctl_file(os.path.join(UMLS_PATH, "MRHIER.ctl"))
hier_df = pd.DataFrame(cleaned, columns=colnames)
hier_df["AB_AUI"] = hier_df["PTR"].map(lambda x: x.split(".")[-2] if len(x.split(".")) >= 2 else "NO_AB")
hier_df = hier_df[["CUI","AUI","PAUI", "AB_AUI"]]
hier_df.drop_duplicates(inplace=True)
hier_df.head()

In [ ]:
del cleaned, lines
gc.collect()

In [ ]:
hier_df["PCUI"] = hier_df.PAUI.map(lambda x: dict_AUI_CUI.get(x,"") )
hier_df["AB_CUI"] = hier_df.AB_AUI.map(lambda x: dict_AUI_CUI.get(x,"") )

In [ ]:
hier_df.head()

In [ ]:
grouped_hier_df =  hier_df.groupby('CUI').agg(lambda x: list(set(x)))

In [ ]:
del hier_df
gc.collect()

In [ ]:
grouped_hier_df.head()

In [ ]:
def remove_empty_values(lst):
    return [x for x in lst if x != '']
grouped_hier_df["PCUI"] = grouped_hier_df.PCUI.map(lambda x: remove_empty_values(x))
grouped_hier_df["AB_CUI"] = grouped_hier_df.AB_CUI.map(lambda x: remove_empty_values(x))

In [ ]:
grouped_hier_df.head()

In [ ]:
cui_pcui_dict = {}
cui_ab_cui_dict = {}

for index, row in grouped_hier_df.iterrows():
    cui = index
    pcui_list = row['PCUI']
    ab_cui_list = row['AB_CUI']

    cui_pcui_dict[cui] = pcui_list

    cui_ab_cui_dict[cui] = ab_cui_list

In [ ]:
grouped_conso_es_td_df = conso_es_td_df.groupby('CUI').agg(lambda x: list(set(x)))
grouped_conso_es_td_df.head()

In [ ]:
del conso_es_td_df, grouped_hier_df
gc.collect()

In [ ]:
grouped_conso_es_td_df.head()

In [ ]:
grouped_conso_es_td_df = grouped_conso_es_td_df.reset_index()
grouped_conso_es_td_df["PCUI"] = grouped_conso_es_td_df.CUI.map(cui_pcui_dict)
grouped_conso_es_td_df["GPCUI"] = grouped_conso_es_td_df.CUI.map(cui_ab_cui_dict)

In [ ]:
grouped_conso_es_td_df = grouped_conso_es_td_df[["CUI","STR","PCUI","GPCUI"]]
grouped_conso_es_td_df.head(25)

In [ ]:
grouped_conso_es_td_df["PCUI"] = grouped_conso_es_td_df["PCUI"].astype(str).map(lambda x: eval(x) if isinstance(x, str) and x!="nan" else [] )
grouped_conso_es_td_df["GPCUI"] = grouped_conso_es_td_df["GPCUI"].astype(str).map(lambda x: eval(x) if isinstance(x, str) and x!="nan" else [] )

In [ ]:
grouped_conso_es_td_df["STR"] = grouped_conso_es_td_df["STR"].map(lambda x: [item.lower() for item in x])

In [ ]:
CUI_STR = dict(zip(grouped_conso_es_td_df.CUI.to_list(), grouped_conso_es_td_df.STR.to_list()))

## Generating Positive Pairs (noparents, parents, and grandparents)

In this section, **positive pairs** are generated to form the triplets that will enrich Representation LLMs. Three types of relations are considered:

- **Noparents**: Concepts without hierarchical relations.
- **Parents**: Directly related concepts (parents).
- **Grandparents**: Two-level related concepts (grandparents).

To avoid **imbalances** in the dataset, the **number of positive pairs per concept is limited**, ensuring a controlled proportion of examples for each type of relation. This balance is crucial to ensure that the model properly learns the different semantic hierarchies.


In [ ]:
noparents_dict = {}
for cui, str_list in zip(grouped_conso_es_td_df['CUI'], grouped_conso_es_td_df['STR']):
    terms = list(set([term for term in str_list if isinstance(term, str)]))
    noparents_dict[cui] = [(term1, term2) for term1, term2 in itertools.combinations(terms, 2)]

In [ ]:
noparents_dict_nmax = {}
for key, value in noparents_dict.items():
    random.shuffle(value)
    noparents_dict_nmax[key] = value[:MAX_NOPARENTS]

In [ ]:
pos_pairs = []
for k,v in tqdm(noparents_dict_nmax.items()):
    for p in v:
        line = str(k) + "||" + p[0] + "||" + p[1]
        pos_pairs.append(line)
len(pos_pairs)

In [ ]:
with open(f'../data/triplets/training_file_umls{UMLS_VERSION.lower()}_es_uncased_no_dup_pairwise_pair_th{MAX_NOPARENTS}_noparents.txt', 'w') as f:
    for line in pos_pairs:
        f.write("%s\n" % line)

In [ ]:
def map_codes(codes):
    return [item for sublist in [CUI_STR.get(code) for code in codes] if sublist for item in sublist]
 
grouped_conso_es_td_df['STR_Parent'] = grouped_conso_es_td_df['PCUI'].map(map_codes)
grouped_conso_es_td_df['STR_Granparent'] = grouped_conso_es_td_df['GPCUI'].map(map_codes)

In [ ]:
grouped_conso_es_td_df.shape

In [ ]:
grouped_conso_es_td_df.head()

In [ ]:
parents_dict = {}
for cui, str_list, str_parlist in zip(grouped_conso_es_td_df['CUI'], grouped_conso_es_td_df['STR'], grouped_conso_es_td_df['STR_Parent']):
    terms = list(set([term for term in str_list if isinstance(term, str)] + str_parlist))
    parents_dict[cui] = [(term1, term2) for term1, term2 in itertools.combinations(terms, 2)]


In [ ]:
parents_dict_nmax = {}
for key, value in parents_dict.items():
    random.shuffle(value)
    parents_dict_nmax[key] = value[:MAX_PARENTS]

In [ ]:
pos_pairs_pcui = []
for k,v in tqdm(parents_dict_nmax.items()):
    for p in v:
        line = str(k) + "||" + p[0] + "||" + p[1]
        pos_pairs_pcui.append(line)
len(pos_pairs_pcui)

In [ ]:
with open(f'../data/triplets/training_file_umls{UMLS_VERSION.lower()}_es_uncased_no_dup_pairwise_pair_th{MAX_PARENTS}_parents.txt', 'w') as f:
    for line in pos_pairs_pcui:
        f.write("%s\n" % line)

In [ ]:
grandparents_dict = {}
for cui, str_list, str_parlist, str_granparent in zip(grouped_conso_es_td_df['CUI'], grouped_conso_es_td_df['STR'], grouped_conso_es_td_df['STR_Parent'], grouped_conso_es_td_df['STR_Granparent']):
    terms = list(set([term for term in str_list if isinstance(term, str)] + str_parlist + str_granparent))
    grandparents_dict[cui] = [(term1, term2) for term1, term2 in itertools.combinations(terms, 2)]

In [ ]:
%%time
grandparents_dict_nmax = {}
for key, value in grandparents_dict.items():
    random.shuffle(value)
    grandparents_dict_nmax[key] = value[:MAX_GRAND_PARENTS]

In [ ]:
pos_pairs_gpcui = []

for k,v in tqdm(grandparents_dict_nmax.items()):
    for p in v:
        line = str(k) + "||" + p[0] + "||" + p[1]
        pos_pairs_gpcui.append(line)

In [ ]:
with open(f'../data/triplets/training_file_umls{UMLS_VERSION.lower()}_es_uncased_no_dup_pairwise_pair_th{MAX_GRAND_PARENTS}_grandparents.txt', 'w') as f:
    for line in pos_pairs_gpcui:
        f.write("%s\n" % line)